In [1]:
import sympy

In [2]:
a, b, c = sympy.symbols("a b c")
r1, r2, r3 = sympy.symbols("r_1 r_2 r_3")
cos_alpha, cos_beta, cos_gamma = sympy.symbols("cos_alpha cos_beta cos_gamma")

In [3]:
def system_to_polynomial(
    system: list[sympy.Expr],
    vars: list[sympy.Symbol],
    keep: list[sympy.Symbol] | None = None,
):
    """
    Convert a polynomial system into a single polynomial via elimination.

    Parameters
    ----------
    system : list of sympy Expr
        Polynomial equations assumed equal to zero.
    vars : list of sympy Symbol
        All variables in the system, ordered for elimination.
    keep : list of sympy Symbol, optional
        Variables to keep (default: keep the first variable only).

    Returns
    -------
    sympy Expr
        Single polynomial in the kept variables.
    """
    vars = list(vars)

    if keep is None:
        keep = [vars[-1]]
    else:
        keep = list(keep)

    eliminate = [v for v in vars if v not in keep]

    # Groebner basis
    G = sympy.groebner(system, *vars, order="lex")

    print(G)

    # Elimination theorem: select polynomials free of eliminated vars
    polys = [
        sympy.factor(g)
        for g in G
        if g.free_symbols.isdisjoint(eliminate)
    ]

    if not polys:
        raise ValueError("Elimination produced no polynomial")

    # Canonical choice: lowest total degree
    polys.sort(key=lambda p: sympy.Poly(p, *keep).total_degree())

    return polys[0]

In [4]:
x, y, z = sympy.symbols('x y z')

system = [
    x + y + z - 1,
    x**2 + y - z,
    y**2 - x,
]

poly_x = system_to_polynomial(system, [x, y, z])
print(poly_x)

GroebnerBasis([2*x - z**3 + 5*z**2 - 12*z + 4, 2*y + z**3 - 5*z**2 + 14*z - 6, z**4 - 6*z**3 + 18*z**2 - 16*z + 4], x, y, z, domain='ZZ', order='lex')
z**4 - 6*z**3 + 18*z**2 - 16*z + 4


In [ ]:
import sympy as sp

# Unknowns
u, v, d1 = sp.symbols("u v d1", real=True)

# Known geometry
a, b, c = sp.symbols("a b c", positive=True)
ca, cb, cg = sp.symbols("cosa cosb cosg", real=True)

# Law-of-cosines
eq1 = (u*d1)**2 + (v*d1)**2 - 2*u*v*d1**2*ca - a**2
eq2 = d1**2 + (v*d1)**2 - 2*v*d1**2*cb - b**2
eq3 = d1**2 + (u*d1)**2 - 2*u*d1**2*cg - c**2

# Remove scale
eq1 /= d1**2
eq2 /= d1**2
eq3 /= d1**2

eq1 = sympy.simplify(eq1)
eq2 = sympy.simplify(eq2)
eq3 = sympy.simplify(eq3)

print(eq1)
print(eq2)
print(eq3)

vars = [v, u]

# Eliminate v → Grunert quartic in u
quartics = system_to_polynomial(
    [eq1, eq2, eq3],
    vars=[v, u],
    keep=[u]
)

assert len(quartics) > 0
quartic = quartics[0]

quartic

-a**2/d1**2 - 2*cosa*u*v + u**2 + v**2
-b**2/d1**2 - 2*cosb*v + v**2 + 1
-c**2/d1**2 - 2*cosg*u + u**2 + 1


In [17]:
import sympy as sp

# Unknowns
u, v = sp.symbols("u v")

# Parameters
a, b, c = sp.symbols("a b c", positive=True)
ca, cb, cg = sp.symbols("ca cb cg", real=True)

# Normalized equations (already divided by d1^2)
eq1 = u**2 + v**2 - 2*u*v*ca - a**2
eq2 = 1 + v**2 - 2*v*cb - b**2
eq3 = 1 + u**2 - 2*u*cg - c**2

# Step 1: eliminate v using eq1 & eq2
res_v = sp.resultant(eq1, eq2, v)

# Step 2: eliminate scale using eq3
quartic = sp.resultant(res_v, eq3, u)

quartic = sp.factor(quartic)
display(quartic)

sp.Poly(quartic, u)

a**8 - 4*a**6*b**2 - 4*a**6*c**2 + 8*a**6*ca*cb*cg - 8*a**6*cb**2 - 8*a**6*cg**2 + 8*a**6 + 6*a**4*b**4 - 8*a**4*b**2*c**2*ca**2 + 12*a**4*b**2*c**2 - 16*a**4*b**2*ca**2*cg**2 + 8*a**4*b**2*ca**2 - 8*a**4*b**2*ca*cb*cg + 16*a**4*b**2*cb**2 + 24*a**4*b**2*cg**2 - 24*a**4*b**2 + 6*a**4*c**4 - 16*a**4*c**2*ca**2*cb**2 + 8*a**4*c**2*ca**2 - 8*a**4*c**2*ca*cb*cg + 24*a**4*c**2*cb**2 + 16*a**4*c**2*cg**2 - 24*a**4*c**2 + 16*a**4*ca**2*cb**2 + 16*a**4*ca**2*cg**2 - 8*a**4*ca**2 - 32*a**4*ca*cb**3*cg - 32*a**4*ca*cb*cg**3 + 16*a**4*ca*cb*cg + 16*a**4*cb**4 + 48*a**4*cb**2*cg**2 - 40*a**4*cb**2 + 16*a**4*cg**4 - 40*a**4*cg**2 + 24*a**4 - 4*a**2*b**6 + 16*a**2*b**4*c**2*ca**2 - 12*a**2*b**4*c**2 + 32*a**2*b**4*ca**2*cg**2 - 16*a**2*b**4*ca**2 - 8*a**2*b**4*ca*cb*cg - 8*a**2*b**4*cb**2 - 24*a**2*b**4*cg**2 + 24*a**2*b**4 + 16*a**2*b**2*c**4*ca**2 - 12*a**2*b**2*c**4 + 32*a**2*b**2*c**2*ca**3*cb*cg - 64*a**2*b**2*c**2*ca**2 + 48*a**2*b**2*c**2*ca*cb*cg - 32*a**2*b**2*c**2*cb**2 - 32*a**2*b**2*c**2

Poly(a**8 - 4*a**6*b**2 - 4*a**6*c**2 + 8*a**6*ca*cb*cg - 8*a**6*cb**2 - 8*a**6*cg**2 + 8*a**6 + 6*a**4*b**4 - 8*a**4*b**2*c**2*ca**2 + 12*a**4*b**2*c**2 - 16*a**4*b**2*ca**2*cg**2 + 8*a**4*b**2*ca**2 - 8*a**4*b**2*ca*cb*cg + 16*a**4*b**2*cb**2 + 24*a**4*b**2*cg**2 - 24*a**4*b**2 + 6*a**4*c**4 - 16*a**4*c**2*ca**2*cb**2 + 8*a**4*c**2*ca**2 - 8*a**4*c**2*ca*cb*cg + 24*a**4*c**2*cb**2 + 16*a**4*c**2*cg**2 - 24*a**4*c**2 + 16*a**4*ca**2*cb**2 + 16*a**4*ca**2*cg**2 - 8*a**4*ca**2 - 32*a**4*ca*cb**3*cg - 32*a**4*ca*cb*cg**3 + 16*a**4*ca*cb*cg + 16*a**4*cb**4 + 48*a**4*cb**2*cg**2 - 40*a**4*cb**2 + 16*a**4*cg**4 - 40*a**4*cg**2 + 24*a**4 - 4*a**2*b**6 + 16*a**2*b**4*c**2*ca**2 - 12*a**2*b**4*c**2 + 32*a**2*b**4*ca**2*cg**2 - 16*a**2*b**4*ca**2 - 8*a**2*b**4*ca*cb*cg - 8*a**2*b**4*cb**2 - 24*a**2*b**4*cg**2 + 24*a**2*b**4 + 16*a**2*b**2*c**4*ca**2 - 12*a**2*b**2*c**4 + 32*a**2*b**2*c**2*ca**3*cb*cg - 64*a**2*b**2*c**2*ca**2 + 48*a**2*b**2*c**2*ca*cb*cg - 32*a**2*b**2*c**2*cb**2 - 32*a**2*b**2

In [ ]:
import sympy as sp

# Unknowns (distances to the 3 points)
d1, d2, d3 = sp.symbols("d1 d2 d3", real=True, positive=True)

# Known world triangle sides
s12, s13, s23 = sp.symbols("s12 s13 s23", positive=True)

# Known angles between bearing vectors
m12, m13, m23 = sp.symbols("m12 m13 m23", real=True)  # cos(alpha), cos(beta), cos(gamma)

# -------------------------------
# 1. Law-of-cosines equations
# -------------------------------
eq1 = d1**2 + d2**2 - 2*d1*d2*m12 - s12**2
eq2 = d1**2 + d3**2 - 2*d1*d3*m13 - s13**2
eq3 = d2**2 + d3**2 - 2*d2*d3*m23 - s23**2

# -------------------------------
# 2. Normalize by d1
# -------------------------------
# Define ratios u = d2/d1, v = d3/d1
u, v = sp.symbols("u v", real=True, positive=True)
subs = {d2: u*d1, d3: v*d1}

eq1_normalized = eq1.subs(subs)
eq2_normalized = eq2.subs(subs)
eq3_normalized = eq3.subs(subs)

eq1_normalized /= d1**2
eq2_normalized /= d1**2
eq3_normalized /= d1**2

# use one of the eqs to get an expression for d1 in terms of u and v
d1sq_expr = sp.solve(eq2_normalized, d1**2)[0]
print(d1sq_expr)

eq1_normalized = eq1_normalized.subs({d1**2: d1sq_expr})
eq3_normalized = eq3_normalized.subs({d1**2: d1sq_expr})

eq1_normalized = sp.simplify(eq1_normalized)
eq3_normalized = sp.simplify(eq3_normalized)

print(eq1_normalized)
print(eq3_normalized)

# -------------------------------
# 3. Eliminate v using resultant to get quartic in u
# -------------------------------
quartic_u = sp.resultant(eq1_normalized, eq3_normalized, v)
quartic_u *= s13**4     # simplify

quartic_u = sp.factor(quartic_u)

print("Grunert quartic in u (symbolic):")
display(quartic_u)
sp.Poly(quartic_u, u)

s13**2/(-2*m13*v + v**2 + 1)
(-2*m12*s13**2*u + s12**2*(2*m13*v - v**2 - 1) + s13**2*u**2 + s13**2)/s13**2
(-2*m23*s13**2*u*v + s13**2*u**2 + s13**2*v**2 + s23**2*(2*m13*v - v**2 - 1))/s13**2
Grunert quartic in u (symbolic):


4*m12**2*s13**4*u**2 - 8*m12**2*s13**2*s23**2*u**2 + 4*m12**2*s23**4*u**2 + 8*m12*m13**2*s12**2*s23**2*u - 8*m12*m13*m23*s12**2*s13**2*u**2 - 8*m12*m13*m23*s12**2*s23**2*u**2 + 8*m12*m23**2*s12**2*s13**2*u**3 - 4*m12*s12**2*s13**2*u**3 + 4*m12*s12**2*s13**2*u + 4*m12*s12**2*s23**2*u**3 - 4*m12*s12**2*s23**2*u - 4*m12*s13**4*u**3 - 4*m12*s13**4*u + 8*m12*s13**2*s23**2*u**3 + 8*m12*s13**2*s23**2*u - 4*m12*s23**4*u**3 - 4*m12*s23**4*u + 4*m13**2*s12**4*u**2 - 4*m13**2*s12**2*s23**2*u**2 - 4*m13**2*s12**2*s23**2 - 4*m13*m23*s12**4*u**3 - 4*m13*m23*s12**4*u + 4*m13*m23*s12**2*s13**2*u**3 + 4*m13*m23*s12**2*s13**2*u + 4*m13*m23*s12**2*s23**2*u**3 + 4*m13*m23*s12**2*s23**2*u + 4*m23**2*s12**4*u**2 - 4*m23**2*s12**2*s13**2*u**4 - 4*m23**2*s12**2*s13**2*u**2 + s12**4*u**4 - 2*s12**4*u**2 + s12**4 + 2*s12**2*s13**2*u**4 - 2*s12**2*s13**2 - 2*s12**2*s23**2*u**4 + 2*s12**2*s23**2 + s13**4*u**4 + 2*s13**4*u**2 + s13**4 - 2*s13**2*s23**2*u**4 - 4*s13**2*s23**2*u**2 - 2*s13**2*s23**2 + s23**4*u**4 + 

Poly((-4*m23**2*s12**2*s13**2 + s12**4 + 2*s12**2*s13**2 - 2*s12**2*s23**2 + s13**4 - 2*s13**2*s23**2 + s23**4)*u**4 + (8*m12*m23**2*s12**2*s13**2 - 4*m12*s12**2*s13**2 + 4*m12*s12**2*s23**2 - 4*m12*s13**4 + 8*m12*s13**2*s23**2 - 4*m12*s23**4 - 4*m13*m23*s12**4 + 4*m13*m23*s12**2*s13**2 + 4*m13*m23*s12**2*s23**2)*u**3 + (4*m12**2*s13**4 - 8*m12**2*s13**2*s23**2 + 4*m12**2*s23**4 - 8*m12*m13*m23*s12**2*s13**2 - 8*m12*m13*m23*s12**2*s23**2 + 4*m13**2*s12**4 - 4*m13**2*s12**2*s23**2 + 4*m23**2*s12**4 - 4*m23**2*s12**2*s13**2 - 2*s12**4 + 2*s13**4 - 4*s13**2*s23**2 + 2*s23**4)*u**2 + (8*m12*m13**2*s12**2*s23**2 + 4*m12*s12**2*s13**2 - 4*m12*s12**2*s23**2 - 4*m12*s13**4 + 8*m12*s13**2*s23**2 - 4*m12*s23**4 - 4*m13*m23*s12**4 + 4*m13*m23*s12**2*s13**2 + 4*m13*m23*s12**2*s23**2)*u - 4*m13**2*s12**2*s23**2 + s12**4 - 2*s12**2*s13**2 + 2*s12**2*s23**2 + s13**4 - 2*s13**2*s23**2 + s23**4, u, domain='ZZ[s12,s13,s23,m12,m13,m23]')